In [1]:
pip install pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import os

folder_path = r"C:\Users\user\Documents\Senior Thesis Excel Files\Alternate Data"

file_names = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]

print(file_names)

['Alt_Analysis.py', 'Alt_Data_cleaning.ipynb', 'Combined Jeonse and Monthly Rent Index.xlsx', 'Integrated Monthly Rent Price Index by Property Type.xlsx', 'Jeonse Price Index by Property Type.xlsx', 'master_panel.xlsx', 'Property Price Index by Type.xlsx', 'Ratio of Jeonse (rental deposit) price to sales price by property type.xlsx']


In [4]:
file_names = [
    "Combined Jeonse and Monthly Rent Index",
    "Integrated Monthly Rent Price Index by Property Type",
    "Jeonse Price Index by Property Type",
    "Property Price Index by Type",
    "Ratio of Jeonse (rental deposit) price to sales price by property type"
]

df_arr = []

for metric_name in file_names:
    file_path = os.path.join(folder_path, f"{metric_name}.xlsx")
    
    df = pd.read_excel(file_path, sheet_name="데이터")
    
    # Rename first two columns
    df = df.rename(columns={
        df.columns[0]: "HousingType",
        df.columns[1]: "Region"
    })
    
    # Fill down merged housing type cells
    df["HousingType"] = df["HousingType"].ffill()
    
    # Reshape wide to long
    df_long = df.melt(
        id_vars=["HousingType", "Region"],
        var_name="Date",
        value_name="Value"
    )
    
    # Convert date
    df_long["Date"] = pd.to_datetime(df_long["Date"].astype(str), format="%Y.%m")
    
    # Add metric column using file name
    df_long["Metric"] = metric_name
    
    # Reorder columns
    df_long = df_long[["Metric", "HousingType", "Region", "Date", "Value"]]
    
    # Sort
    df_long = df_long.sort_values(["HousingType", "Region", "Date"]).reset_index(drop=True)
    
    # Store in array in same order as file_names
    df_arr.append(df_long)

print(df_arr[0].head())

housing_df_arr = []

for df_long in df_arr:
    housing_dict = {}
    
    for housing_type in df_long["HousingType"].unique():
        housing_dict[housing_type] = (
            df_long[df_long["HousingType"] == housing_type]
            .copy()
            .reset_index(drop=True)
        )
    
    housing_df_arr.append(housing_dict)

                                   Metric HousingType        Region  \
0  Combined Jeonse and Monthly Rent Index   Apartment  Central Area   
1  Combined Jeonse and Monthly Rent Index   Apartment  Central Area   
2  Combined Jeonse and Monthly Rent Index   Apartment  Central Area   
3  Combined Jeonse and Monthly Rent Index   Apartment  Central Area   
4  Combined Jeonse and Monthly Rent Index   Apartment  Central Area   

        Date  Value  
0 2015-06-01   90.9  
1 2015-07-01   91.1  
2 2015-08-01   91.3  
3 2015-09-01   91.5  
4 2015-10-01   91.9  


In [7]:
master_df = pd.concat(df_arr, ignore_index=True)
# master_df.to_excel(os.path.join(folder_path, "master_panel.xlsx"), index=False)

In [8]:
import pandas as pd
import numpy as np

# Combine all metric-level long dataframes into one master dataframe
master_df = pd.concat(df_arr, ignore_index=True)

# Pivot so that each metric becomes its own column
panel_df = master_df.pivot_table(
    index=["Date", "HousingType", "Region"],
    columns="Metric",
    values="Value"
).reset_index()

import pandas as pd
import numpy as np

# Combine all metric-level long dataframes into one master dataframe
master_df = pd.concat(df_arr, ignore_index=True)

# Pivot so that each metric becomes its own column
panel_df = master_df.pivot_table(
    index=["Date", "HousingType", "Region"],
    columns="Metric",
    values="Value"
).reset_index()

# Optional: flatten columns if needed
panel_df.columns.name = None

tightening_dates = pd.to_datetime(["2017.08", "2018.09", "2019.12"], format="%Y.%m")
loosening_dates  = pd.to_datetime(["2022.12"], format="%Y.%m")

metric_rename_map = {
    file_names[0]: "Combined Jeonse and Monthly Rent Index",
    file_names[1]: "Integrated Monthly Rent Price Index",
    file_names[2]: "Jeonse Price Index",
    file_names[3]: "Property Price Index",
    file_names[4]: "Price/Jeonse"
}

panel_df = panel_df.rename(columns=metric_rename_map)

required_cols = list(metric_rename_map.values())

# Sort before grouped calculations
panel_df = panel_df.sort_values(["HousingType", "Region", "Date"]).reset_index(drop=True)

# Drop rows missing the required base inputs
panel_df = panel_df.dropna(subset=required_cols).copy()

# Compute log-normalized components within each HousingType × Region series
group_cols = ["HousingType", "Region"]

panel_df["Component Non-Ownership"] = panel_df.groupby(group_cols)["Combined Jeonse and Monthly Rent Index"].transform(
    lambda x: np.log(x / x.iloc[0])
)

panel_df["Component Jeonse"] = panel_df.groupby(group_cols)["Jeonse Price Index"].transform(
    lambda x: np.log(x / x.iloc[0])
)

panel_df["Component Rent"] = panel_df.groupby(group_cols)["Integrated Monthly Rent Price Index"].transform(
    lambda x: np.log(x / x.iloc[0])
)

panel_df["Component Price"] = panel_df.groupby(group_cols)["Property Price Index"].transform(
    lambda x: np.log(x / x.iloc[0])
)

panel_df["Price/Nonownership Ratio"] = panel_df["Component Price"] - panel_df["Component Non-Ownership"]
panel_df["Price/Rent"] = panel_df["Component Price"] - panel_df["Component Rent"]
panel_df["Price/Jeonse"] = panel_df.groupby(["HousingType", "Region"])["Price/Jeonse"].transform(
    lambda x: np.log(x.iloc[0] / x)
)

# Policy dummy
panel_df["Policy Dummy"] = 0
panel_df.loc[panel_df["Date"].isin(tightening_dates), "Policy Dummy"] = 1
panel_df.loc[panel_df["Date"].isin(loosening_dates), "Policy Dummy"] = -1

category_names = [
    "Comprehensive Housing Index",
    "Apartment",
    "Low-Rise Multi-Unit Housing",
    "Single-Family House"
]

final_df_arr = []

for category in category_names:
    temp_df = panel_df[panel_df["HousingType"] == category].copy()
    temp_df = temp_df.sort_values(["Region", "Date"]).reset_index(drop=True)
    final_df_arr.append(temp_df)

In [9]:
import os

output_path = os.path.join(folder_path, "master_panel.xlsx")

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    panel_df.to_excel(writer, sheet_name="master_panel", index=False)

print(f"Saved: {output_path}")

Saved: C:\Users\user\Documents\Senior Thesis Excel Files\Alternate Data\master_panel.xlsx
